# Bounding Box GPKG Inspection

Clean notebook to inspect `bounding_box.gpkg` with explicit path configuration.

In [1]:
# Optional: install once in this kernel
# %pip install -q geopandas pyogrio pyproj shapely matplotlib pandas

In [2]:
from pathlib import Path
import os

# ---- Edit only these paths ----
PROJECT_ROOT = Path('/lustre06/project/6007330/keyuyao/power-grid-detection')
GPKG_PATH = PROJECT_ROOT / 'data' / 'annotations' / 'bounding_box.gpkg'
PROJ_PATH = Path('/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Compiler/gcccore/proj/9.4.1/share/proj')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('GPKG_PATH:', GPKG_PATH)
print('PROJ_PATH:', PROJ_PATH)

if not GPKG_PATH.exists():
    raise FileNotFoundError(f'Missing geopackage: {GPKG_PATH}')
if not (PROJ_PATH / 'proj.db').exists():
    raise FileNotFoundError(f'Missing proj.db at: {PROJ_PATH}')

PROJECT_ROOT: /lustre06/project/6007330/keyuyao/power-grid-detection
GPKG_PATH: /lustre06/project/6007330/keyuyao/power-grid-detection/data/annotations/bounding_box.gpkg
PROJ_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Compiler/gcccore/proj/9.4.1/share/proj


In [3]:
# Configure PROJ before importing geopandas/pyogrio
os.environ['PROJ_DATA'] = str(PROJ_PATH)
os.environ['PROJ_LIB'] = str(PROJ_PATH)

from pyproj import datadir
datadir.set_data_dir(str(PROJ_PATH))
print('Using PROJ data dir:', datadir.get_data_dir())

Using PROJ data dir: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Compiler/gcccore/proj/9.4.1/share/proj


In [4]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [5]:
layers = gpd.list_layers(GPKG_PATH)
display(layers)

if layers.empty:
    raise ValueError('No layers found in geopackage')

LAYER_NAME = layers.iloc[0]['name']
print('Selected layer:', LAYER_NAME)

,name,geometry_type
0,bounding_boxes,Polygon


Selected layer: bounding_boxes


In [6]:
gdf = gpd.read_file(GPKG_PATH, layer=LAYER_NAME)
print(f'Rows: {len(gdf):,}')
print('CRS:', gdf.crs)
display(gdf.head(3))
display(gdf.dtypes.to_frame('dtype'))

Rows: 239
CRS: EPSG:2950


,class,width,height,area,perimeter,geometry
0,tower,19.981989,40.120087,801.679149,120.204153,"POLYGON ((250239.66 5014772.839, 250259.642 50..."
1,tower,29.076470,46.266333,1345.261662,150.685607,"POLYGON ((250928.936 5014789.946, 250958.013 5..."
2,tower,28.959573,44.303824,1283.019814,146.526794,"POLYGON ((251632.871 5014812.561, 251661.831 5..."


,dtype
class,str
width,float64
height,float64
area,float64
perimeter,float64
geometry,geometry


In [7]:
qa = {
    'rows': int(len(gdf)),
    'null_geometry': int(gdf.geometry.isna().sum()),
    'empty_geometry': int(gdf.geometry.is_empty.sum()),
    'invalid_geometry': int((~gdf.geometry.is_valid).sum()),
}
qa['valid_geometry'] = qa['rows'] - qa['null_geometry'] - qa['empty_geometry'] - qa['invalid_geometry']
display(pd.Series(qa).to_frame('count'))

,count
rows,239
null_geometry,0
empty_geometry,0
invalid_geometry,0
valid_geometry,239


In [10]:
candidate_label_cols = ['class', 'label', 'category', 'category_id', 'name', 'type']
label_col = next((c for c in candidate_label_cols if c in gdf.columns), None)
print('Detected label column:', label_col)

if label_col:
    display(gdf[label_col].astype(str).value_counts(dropna=False).head(30).to_frame('count'))
else:
    print('No common label column found. Available columns:')
    print(list(gdf.columns))

Detected label column: class


,count
class,
tower,232
wind_turbine,5
substation,1
tower,1
